In [ ]:
from flask import Flask, request
from datetime import datetime, date
import pandas as pd
import requests
import smtplib
import os
import io
import base64
import matplotlib.pyplot as plt
from email.message import EmailMessage
from dotenv import load_dotenv
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import nest_asyncio
from werkzeug.serving import run_simple

load_dotenv()

app = Flask(__name__)


# Email Config

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")


# Global storage

latest_alerts_df = pd.DataFrame()
latest_weather_df = pd.DataFrame()
latest_city = ""
latest_freq = ""
latest_start_date = None
latest_end_date = None



# Helper Functions

def get_coords(city):
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    geo_response = requests.get(geo_url).json()

    if "results" not in geo_response or not geo_response["results"]:
        raise ValueError("City not found")

    lat = geo_response["results"][0]["latitude"]
    lon = geo_response["results"][0]["longitude"]
    return lat, lon


def fetchtime(date_val):
    return date_val.split("T")[1] if "T" in date_val else None


def fetchdate(date_val):
    return date_val.split("T")[0] if "T" in date_val else None


def fig_to_base64():
    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    img_base64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    buf.close()
    plt.close()
    return img_base64


def get_weather_alerts(city, freq, start_date, end_date):
    lat, lon = get_coords(city)

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,windspeed_10m,precipitation,weathercode",
        "daily": "temperature_2m_max,temperature_2m_min,windspeed_10m_max,precipitation_sum,weathercode",
        "timezone": "auto"
    }

    url = "https://archive-api.open-meteo.com/v1/archive"
    response = requests.get(url, params=params)
    data = response.json()

    HIGH_WIND = 15
    HEAVY_RAIN = 10
    HIGH_TEMP = 35
    LOW_TEMP = -5

    if freq == "hourly":
        if "hourly" not in data:
            raise ValueError("Hourly weather data not available.")

        df = pd.DataFrame(data["hourly"])
        df["hour"] = df["time"].apply(fetchtime)
        df["date"] = df["time"].apply(fetchdate)
        df["temperature_2m"] = df["temperature_2m"].astype(float)
        df["windspeed_10m"] = df["windspeed_10m"].astype(float)
        df["precipitation"] = df["precipitation"].astype(float)

        df = df.rename(columns={
            "temperature_2m": "Temp",
            "windspeed_10m": "Wind",
            "precipitation": "Rain"
        })

        # Alerts
        df["alert"] = None
        df.loc[df["Wind"] >= HIGH_WIND, "alert"] = "High wind"
        df.loc[df["Rain"] >= HEAVY_RAIN, "alert"] = "Heavy rain"
        df.loc[df["Temp"] >= HIGH_TEMP, "alert"] = "High temperature"
        df.loc[df["Temp"] <= LOW_TEMP, "alert"] = "Low temperature"

        df = df[["date", "hour", "Temp", "Wind", "Rain", "alert"]]

    else:
        if "daily" not in data:
            raise ValueError("Daily weather data not available.")

        df = pd.DataFrame(data["daily"])
        df["Temp"] = (df["temperature_2m_max"] + df["temperature_2m_min"]) / 2
        df["Max Wind"] = df["windspeed_10m_max"].astype(float)
        df["Total Rain"] = df["precipitation_sum"].astype(float)
        df["date"] = df["time"]

        # Alerts
        df["alert"] = None
        df.loc[df["Max Wind"] >= HIGH_WIND, "alert"] = "High wind"
        df.loc[df["Total Rain"] >= HEAVY_RAIN, "alert"] = "Heavy rain"
        df.loc[df["Temp"] >= HIGH_TEMP, "alert"] = "High temperature"
        df.loc[df["Temp"] <= LOW_TEMP, "alert"] = "Low temperature"

        df = df[["date", "Temp", "Max Wind", "Total Rain", "alert"]]

    alerts_df = df[df["alert"].notnull()].copy()
    return df, alerts_df


def create_regression_graph(x, y, x_label, city):
    plt.figure(figsize=(14, 5))
    plt.scatter(x, y, alpha=0.7, label="Data points")

    model = LinearRegression()
    X_line = x.values.reshape(-1, 1)
    model.fit(X_line, y)
    y_line = model.predict(X_line)

    sorted_idx = x.argsort()
    plt.plot(x.iloc[sorted_idx], y_line[sorted_idx], linewidth=2, label="Regression line")

    plt.xlabel(x_label)
    plt.ylabel("Temperature")
    plt.title(f"{city} Temperature vs {x_label}")
    plt.grid(True, alpha=0.3)
    plt.legend()

    return fig_to_base64()


def create_graphs(df, city, freq):
    graphs = {}

    if freq == "hourly":
        # Graph 1: Temperature and Windspeed
        plt.figure(figsize=(14, 5))
        plt.plot(df["hour"], df["Temp"], label="Temperature (°C)")
        plt.plot(df["hour"], df["Wind"], label="Windspeed (m/s)")
        plt.xlabel("Hour")
        plt.ylabel("Values")
        plt.title(f"{city} Hourly Temperature and Windspeed")
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.legend()
        graphs["temp_wind"] = fig_to_base64()

        # Graph 2: Precipitation
        plt.figure(figsize=(14, 5))
        plt.plot(df["hour"], df["Rain"])
        plt.xlabel("Hour")
        plt.ylabel("Precipitation (mm)")
        plt.title(f"{city} Hourly Rainfall")
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        graphs["precipitation"] = fig_to_base64()

        # Graph 3: Regression plot
        graphs["regplot"] = create_regression_graph(
            df["Wind"], df["Temp"], "Wind", city
        )

    else:
        # Graph 1: Temperature and Windspeed
        plt.figure(figsize=(14, 5))
        plt.plot(df["date"], df["Temp"], label="Temperature (°C)")
        plt.plot(df["date"], df["Max Wind"], label="Windspeed (m/s)")
        plt.xlabel("Date")
        plt.ylabel("Values")
        plt.title(f"{city} Daily Temperature and Windspeed")
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.legend()
        graphs["temp_wind"] = fig_to_base64()

        # Graph 2: Precipitation
        plt.figure(figsize=(14, 5))
        plt.plot(df["date"], df["Total Rain"])
        plt.xlabel("Date")
        plt.ylabel("Precipitation (mm)")
        plt.title(f"{city} Daily Precipitation")
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        graphs["precipitation"] = fig_to_base64()

        # Graph 3: Regression plot
        graphs["regplot"] = create_regression_graph(
            df["Max Wind"], df["Temp"], "Max Wind", city
        )

    return graphs


def run_regression(df):
    if "Wind" in df.columns:
        X = df[["Wind", "Rain"]]
    else:
        X = df[["Max Wind", "Total Rain"]]

    y = df["Temp"]

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)

    return {
        "r2": round(r2, 3),
        "coef_wind": round(model.coef_[0], 2),
        "coef_precip": round(model.coef_[1], 2),
    }


def alerts_df_to_text(alerts_df, city, freq, start_date, end_date):
    if alerts_df.empty:
        return (
            f"Weather Alert Report\n\n"
            f"City: {city}\n"
            f"Frequency: {freq}\n"
            f"Start Date: {start_date}\n"
            f"End Date: {end_date}\n\n"
            f"No upcoming weather alerts based on thresholds."
        )

    if freq == "hourly":
        cols_to_show = ["date", "hour", "Temp", "Wind", "Rain", "alert"]
    else:
        cols_to_show = ["date", "Temp", "Max Wind", "Total Rain", "alert"]

    report_df = alerts_df[cols_to_show]

    return (
        f"Weather Alert Report\n\n"
        f"City: {city}\n"
        f"Frequency: {freq}\n"
        f"Start Date: {start_date}\n"
        f"End Date: {end_date}\n\n"
        f"Upcoming Weather Alerts:\n\n"
        f"{report_df.to_string(index=False)}"
    )


def answer_stats_question(question, df, city, freq):
    if df.empty:
        return "Please generate weather data first from the home page."

    q = question.lower().strip()

    try:
        if freq == "hourly":
            if "average temperature" in q or "temperature" in q or "mean temperature" in q:
                return f"The average temperature in {city} is {df['Temp'].mean():.2f} °C."

            elif "maximum temperature" in q or "max temperature" in q:
                row = df.loc[df["Temp"].idxmax()]
                return f"The maximum temperature is {row['Temp']:.2f} °C on {row['date']} at {row['hour']}."

            elif "minimum temperature" in q or "min temperature" in q:
                row = df.loc[df["Temp"].idxmin()]
                return f"The minimum temperature is {row['Temp']:.2f} °C on {row['date']} at {row['hour']}."

            elif "average wind" in q or "wind" in q or "mean wind" in q:
                return f"The average wind speed in {city} is {df['Wind'].mean():.2f} m/s."

            elif "maximum wind" in q or "max wind" in q:
                row = df.loc[df["Wind"].idxmax()]
                return f"The maximum wind speed is {row['Wind']:.2f} m/s on {row['date']} at {row['hour']}."

            elif "total rain" in q or "rain" in q or "total precipitation" in q:
                return f"The total precipitation is {df['Rain'].sum():.2f} mm."

            elif "alerts" in q  or "alert" in q:
                alerts_df = df[df["alert"].notna()]

                count = len(alerts_df)

                return (
                    f"There are {count} weather alert records in the current data.\n\n"
                    + alerts_df.to_html(index=False)
                )

            elif "r2" in q or "regression" in q:
                stats = run_regression(df)
                return (
                    f"The regression R² is {stats['r2']}. "
                    f"Wind coefficient is {stats['coef_wind']}, "
                    f"and precipitation coefficient is {stats['coef_precip']}."
                )

            elif "how many rows" in q or "how many records" in q:
                return f"There are {len(df)} hourly weather records."

            else:
                return (
                    "I can answer questions like average temperature, max temperature, "
                    "min temperature, average wind, max wind, total rain, how many alerts, "
                    "regression stats, and number of records."
                )

        else:
            if "average temperature" in q or "temperature" in q or "mean temperature" in q:
                return f"The average temperature in {city} is {df['Temp'].mean():.2f} °C."

            elif "maximum temperature" in q or "max temperature" in q:
                row = df.loc[df["Temp"].idxmax()]
                return f"The maximum temperature is {row['Temp']:.2f} °C on {row['date']}."

            elif "minimum temperature" in q or "min temperature" in q:
                row = df.loc[df["Temp"].idxmin()]
                return f"The minimum temperature is {row['Temp']:.2f} °C on {row['date']}."

            elif "average wind" in q or "wind" in q or "mean wind" in q:
                return f"The average maximum wind speed in {city} is {df['Max Wind'].mean():.2f} m/s."

            elif "maximum wind" in q or "max wind" in q:
                row = df.loc[df["Max Wind"].idxmax()]
                return f"The maximum wind speed is {row['Max Wind']:.2f} m/s on {row['date']}."

            elif "total rain" in q or "rain" in q or "total precipitation" in q:
                return f"The total precipitation is {df['Total Rain'].sum():.2f} mm."

            elif "how many alerts" in q or "alerts" in q or "alert" in q:
                alerts_df = df[df["alert"].notna()]

                count = len(alerts_df)

                return (
                    f"There are {count} weather alert records in the current data.\n\n"
                    + alerts_df.to_html(index=False)
                )

            elif "r2" in q or "regression" in q:
                stats = run_regression(df)
                return (
                    f"The regression R² is {stats['r2']}. "
                    f"Wind coefficient is {stats['coef_wind']}, "
                    f"and precipitation coefficient is {stats['coef_precip']}."
                )

            elif "how many rows" in q or "how many records" in q:
                return f"There are {len(df)} daily weather records."

            else:
                return (
                    "I can answer questions like average temperature, max temperature, "
                    "min temperature, average wind, max wind, total rain, how many alerts, "
                    "regression stats, and number of records."
                )

    except Exception as e:
        return f"Error analyzing weather statistics: {str(e)}"



# Routes

@app.route("/", methods=["GET", "POST"])
def home():
    global latest_alerts_df, latest_weather_df, latest_city, latest_freq, latest_start_date, latest_end_date

    if request.method == "POST":
        try:
            city = request.form.get("city")
            freq = request.form.get("frequency")
            date_input = request.form.get("date")

            start_date = datetime.strptime(date_input, "%Y-%m-%d").date()

            if freq == "hourly":
                end_date = start_date
            elif freq == "daily":
                end_date = date.today()
            else:
                freq = "hourly"
                end_date = start_date

            df, alerts_df = get_weather_alerts(city, freq, start_date, end_date)
            graphs = create_graphs(df, city, freq)
            regression = run_regression(df)

            latest_weather_df = df.copy()
            latest_alerts_df = alerts_df
            latest_city = city
            latest_freq = freq
            latest_start_date = start_date
            latest_end_date = end_date

            display_df = df.copy()

            display_df = display_df.rename(columns={
                "Temp": "Temperature (°C)",
                "Wind": "Wind Speed (m/s)",
                "Rain": "Precipitation (mm)",
                "Max Wind": "Max Wind Speed (m/s)",
                "Total Rain": "Total Precipitation (mm)",
                "date": "Date",
                "hour": "Hour",
                "alert": "Alert"
            })

            table_html = display_df.to_html(index=False, border=1)

            return f"""
            <html>
            <head>
                <title>Weather App</title>
                <style>
                    body {{ font-family: Arial; margin: 40px; background-color: #f4f4f4; }}
                    .container {{
                        position: relative;
                        background: white;
                        padding: 20px;
                        border-radius: 10px;
                        width: 95%;
                        max-width: 1200px;
                    }}
                    input, select {{ width: 100%; padding: 8px; margin: 10px 0; }}
                    button, .btn {{
                        padding: 10px 15px;
                        background: #3498db;
                        color: white;
                        border: none;
                        text-decoration: none;
                        display: inline-block;
                        margin-top: 10px;
                        border-radius: 5px;
                        margin-right: 8px;
                    }}
                    table {{
                        border-collapse: collapse;
                        width: 100%;
                        margin-top: 15px;
                    }}
                    th, td {{
                        padding: 8px;
                        border: 1px solid #ddd;
                        text-align: left;
                    }}
                    th {{
                        background: #3498db;
                        color: white;
                    }}
                    img {{
                        width: 100%;
                        max-width: 1000px;
                        margin-top: 20px;
                        border: 1px solid #ccc;
                        border-radius: 8px;
                    }}
                    .section {{
                        margin-top: 30px;
                    }}
                    .top-right-btn {{
                        position: absolute;
                        top: 20px;
                        right: 20px;
                    }}
                </style>
            </head>
            <body>
                <div class="container">
                    <a class="btn" href="/">Go Back</a>
                    <a class="btn" href="/stats-chat">Stats Chat</a>
                    <a class="btn top-right-btn" href="/notify">Send Alerts via Email</a>

                    <h1>Weather Report</h1>
                    <p><b>City:</b> {city}</p>
                    <p><b>Frequency:</b> {freq}</p>
                    <p><b>Start Date:</b> {start_date}</p>
                    <p><b>End Date:</b> {end_date}</p>

                    <div class="section">
                        <h2>Detected Alerts</h2>
                        {table_html}
                    </div>

                    <div class="section">
                        <h2>Temperature and Windspeed</h2>
                        <img src="data:image/png;base64,{graphs['temp_wind']}" />
                    </div>

                    <div class="section">
                        <h2>Precipitation</h2>
                        <img src="data:image/png;base64,{graphs['precipitation']}" />
                    </div>

                    <div class="section">
                        <h2>Temperature vs Windspeed Regression</h2>
                        <img src="data:image/png;base64,{graphs['regplot']}" />
                    </div>

                    <div class="section">
                        <h2>Further Statistics</h2>
                        <p><b>R²:</b> {regression["r2"]}</p>
                        <p><b>Coefficient for Windspeed:</b> {regression["coef_wind"]}</p>
                        <p><b>Coefficient for Precipitation:</b> {regression["coef_precip"]}</p>
                    </div>

                    <a class="btn" href="/">Go Back</a>
                </div>
            </body>
            </html>
            """

        except Exception as e:
            return f"<h3>Error in Programming Logic: {str(e)}</h3><a href='/'>Go Back</a>"

    return """
    <html>
    <head>
        <title>Weather App</title>
        <style>
            body { font-family: Arial; margin: 40px; background-color: #f4f4f4; }
            .container { background: white; padding: 20px; border-radius: 10px; width: 400px; }
            input, select { width: 100%; padding: 8px; margin: 10px 0; }
            button { padding: 10px; background: #3498db; color: white; border: none; border-radius: 5px; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Weather App</h1>

            <form method="POST">
                <label>City</label>
                <input type="text" name="city" required>

                <label>Frequency</label>
                <select name="frequency">
                    <option value="hourly">Hourly</option>
                    <option value="daily">Daily</option>
                </select>

                <label>Date</label>
                <input type="date" name="date" required>

                <button type="submit">Get Weather</button>
            </form>
        </div>
    </body>
    </html>
    """


@app.route("/stats-chat", methods=["GET", "POST"])
def stats_chat():
    global latest_weather_df, latest_city, latest_freq, latest_start_date, latest_end_date

    answer = ""

    if request.method == "POST":
        question = request.form.get("question", "")
        answer = answer_stats_question(
            question,
            latest_weather_df,
            latest_city,
            latest_freq
        )

    current_info = ""
    if latest_start_date is not None:
        current_info = f"""
        <p><b>City:</b> {latest_city}</p>
        <p><b>Frequency:</b> {latest_freq}</p>
        <p><b>Start Date:</b> {latest_start_date}</p>
        <p><b>End Date:</b> {latest_end_date}</p>
        """

    return f"""
    <html>
    <head>
        <title>Weather Stats Chat</title>
        <style>
            body {{ font-family: Arial; margin: 40px; background-color: #f4f4f4; }}
            .container {{
                background: white;
                padding: 20px;
                border-radius: 10px;
                width: 800px;
                position: relative;
            }}
            input {{
                width: 100%;
                padding: 10px;
                margin: 10px 0;
            }}
            button, .btn {{
                padding: 10px 15px;
                background: #3498db;
                color: white;
                border: none;
                text-decoration: none;
                display: inline-block;
                margin-top: 10px;
                border-radius: 5px;
                margin-right: 8px;
            }}
            .answer-box {{
                margin-top: 20px;
                padding: 15px;
                background: #f9f9f9;
                border: 1px solid #ddd;
                border-radius: 8px;
            }}
            .examples {{
                margin-top: 15px;
                background: #eef6ff;
                padding: 12px;
                border-radius: 8px;
            }}
            .top-links {{
                position: absolute;
                top: 20px;
                right: 20px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="top-links">
                <a class="btn" href="/">Go Back</a>
                <a class="btn" href="/notify">Notify</a>
            </div>

            <h1>Weather Statistics Chat</h1>
            <p>Ask questions about the currently loaded weather data.</p>

            {current_info}

            <form method="POST">
                <label>Enter your question</label>
                <input type="text" name="question" placeholder="e.g. What is the average temperature?" required>
                <button type="submit">Ask</button>
            </form>

            <div class="examples">
                <b>Example questions:</b><br>
                What is the average temperature?<br>
                What is the maximum wind?<br>
                What is the total rain?<br>
                How many alerts are there?<br>
                What is the regression R2?<br>
                How many records are loaded?
            </div>

            {"<div class='answer-box'><b>Answer:</b><br><br>" + answer + "</div>" if answer else ""}
        </div>
    </body>
    </html>
    """


@app.route("/notify", methods=["GET", "POST"])
def notify():
    global latest_alerts_df, latest_city, latest_freq, latest_start_date, latest_end_date

    status = ""

    if request.method == "POST":
        try:
            receiver = request.form.get("receiver")

            if latest_start_date is None:
                raise ValueError("No weather data found. Please generate weather data first.")

            subject = f"Weather Alerts for {latest_city}"
            message = alerts_df_to_text(
                latest_alerts_df,
                latest_city,
                latest_freq,
                latest_start_date,
                latest_end_date
            )

            email_msg = EmailMessage()
            email_msg["Subject"] = subject
            email_msg["From"] = EMAIL_ADDRESS
            email_msg["To"] = receiver
            email_msg.set_content(message)

            with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
                server.starttls()
                server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
                server.send_message(email_msg)

            status = "Email sent successfully!"

        except Exception as e:
            status = f"Error: {str(e)}"

    preview_message = ""
    if latest_start_date is not None:
        preview_message = alerts_df_to_text(
            latest_alerts_df,
            latest_city,
            latest_freq,
            latest_start_date,
            latest_end_date
        )

    return f"""
    <html>
    <head>
        <title>Notifications</title>
        <style>
            body {{ font-family: Arial; margin: 40px; background-color: #f4f4f4; }}
            .container {{ background: white; padding: 20px; border-radius: 10px; width: 700px; }}
            input, textarea {{ width: 100%; padding: 10px; margin: 10px 0; }}
            button {{
                padding: 10px;
                background: #27ae60;
                color: white;
                border: none;
                border-radius: 5px;
            }}
            pre {{
                background: #f9f9f9;
                padding: 15px;
                border: 1px solid #ddd;
                overflow-x: auto;
                white-space: pre-wrap;
            }}
            a {{
                display: inline-block;
                margin-top: 10px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Send Weather Alerts Email</h1>

            <form method="POST">
                <label>Receiver Email</label>
                <input type="email" name="receiver" placeholder="example@gmail.com" required>

                <label>Email Preview</label>
                <pre>{preview_message}</pre>

                <button type="submit">Send Email</button>
            </form>

            <p>{status}</p>
            <a href="/">Go Back</a>
        </div>
    </body>
    </html>
    """


if __name__ == "__main__":
    nest_asyncio.apply()
    run_simple("localhost", 5001, app, use_reloader=False, use_debugger=True)

 * Running on http://localhost:5001
Press CTRL+C to quit
127.0.0.1 - - [01/Apr/2026 16:34:20] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:34:32] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:34:46] "GET /stats-chat HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:34:51] "POST /stats-chat HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:34:54] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:35:05] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:35:06] "GET /stats-chat HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:35:08] "GET /notify HTTP/1.1" 200 -
127.0.0.1 - - [01/Apr/2026 16:35:14] "GET / HTTP/1.1" 200 -
